# 三维高斯泼溅：从参数到像素的中文源码精读

这份 notebook 配合当前 fork 中的中文注释使用，目标是能够解释每个核心张量从哪里来、怎样参与渲染、梯度怎样返回，以及高斯数量为什么会改变。

**对应版本**：主仓库 `54c035f7834b564019656c3e3fcc3646292f727d`；光栅器子模块 `9c5c2028f6fbee2be239bc4c9421ff894fe4fbe0`。这是包含深度正则、曝光补偿、抗锯齿等后续功能的版本，并非只包含论文最初实现。

**运行方式**：所有代码单元只使用 Python 标准库，无须显卡、PyTorch、数据集或网络。请从上至下运行。代码标识符保留常用数学/源码名称，说明、注释、输出均使用中文。这里的小实验帮助推导公式；它们不是原生 CUDA 光栅器，也不是完整训练复现。

**阅读路线**：数据与相机 → 高斯参数 → 球谐颜色 → 协方差投影 → 分块排序 → 透明度融合 → 反向传播 → 增密裁剪 → 训练循环。读完每节再打开链接对应的源码，不必一开始就追全部 C++ 内核。

版权与使用范围沿用各原始文件的许可；源码中文注释不改变原算法。


## 一、定位仓库与核心入口

下面的代码从当前目录向父目录查找仓库，因此既可从仓库根目录启动，也可从本 notebook 的目录启动。它只读取源码，不导入会要求 CUDA 的训练模块。

| 文件 | 要解决的问题 |
|---|---|
| `scene/dataset_readers.py` | 图像、点云、相机外参怎样变成场景信息？ |
| `scene/cameras.py` | 世界到相机、投影矩阵怎样存储？ |
| `scene/gaussian_model.py` | 什么是可训练参数，怎样增密和裁剪？ |
| `gaussian_renderer/__init__.py` | 哪些张量进入 CUDA，哪些张量用于增密？ |
| `train.py` | 一次迭代的真实执行顺序是什么？ |
| `learning/zh-CN/cuda-source/` | 对应版本的 CUDA 前向、反向与调度怎样工作？ |


In [1]:
from pathlib import Path
import ast
import math
import random
import struct

# 向父目录查找，避免依赖某个人机器上的绝对路径。
repo = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / 'train.py').is_file() and (p / 'scene/gaussian_model.py').is_file()), None)
assert repo is not None, '请从仓库根目录或本笔记所在目录启动。'
print('已找到仓库：', repo.name)

def show_source(relative_path, keyword, before=3, after=12):
    # 现场定位代码行，中文注释增删后也不会依赖过期的固定行号。
    lines = (repo / relative_path).read_text(encoding='utf-8').splitlines()
    index = next(i for i, line in enumerate(lines) if keyword in line)
    for i in range(max(0, index - before), min(len(lines), index + after)):
        print(f'{i + 1:4d}  {lines[i]}')

show_source('train.py', 'def training(', before=4, after=8)


已找到仓库： gaussian-splatting
  50      SPARSE_ADAM_AVAILABLE = False
  51  
  52  # 训练入口的三组配置各司其职：dataset 管数据/模型目录，opt 管学习率/迭代，pipe 管渲染实现。
  53  # 三个 iterations 列表分别控制评估、导出场景、保存优化器检查点；它们不是三段训练。
  54  def training(dataset, opt, pipe, testing_iterations, saving_iterations, checkpoint_iterations, checkpoint, debug_from):
  55  
  56      # 显式要求稀疏优化器却没有对应扩展时立即停止，避免悄悄使用不同优化器。
  57      if not SPARSE_ADAM_AVAILABLE and opt.optimizer_type == "sparse_adam":
  58          sys.exit(f"Trying to use sparse adam but it is not installed, please install the correct rasterizer using pip install [3dgs_accel].")
  59  
  60      # 记录恢复训练的起点；从头训练时下方会加一，所以实际迭代编号从 1 开始。
  61      first_iter = 0


## 二、一个高斯不是一个普通三维点

源码：[高斯模型](../../scene/gaussian_model.py)，先读 `__init__`、`setup_functions` 与各 `get_*` 属性。

每个高斯有位置 $\mu\in\mathbb{R}^3$、三维尺度 $s$、旋转四元数 $q$、不透明度 $o$ 和球谐颜色系数。它投影后覆盖一个椭圆区域，多个高斯共同解释照片。这里没有让神经网络为每个采样点预测颜色的步骤。

| 原始可训练参数 | 形状 | 渲染前的转换 |
|---|---|---|
| `_xyz` | $N\times3$ | 直接作为世界坐标 |
| `_scaling` | $N\times3$ | `exp`，确保尺度为正 |
| `_rotation` | $N\times4$ | 归一化，顺序为 $(w,x,y,z)$ |
| `_opacity` | $N\times1$ | `sigmoid`，得到零到一之间的不透明度 |
| `_features_dc` | $N\times1\times3$ | 零阶球谐系数 |
| `_features_rest` | $N\times(K-1)\times3$ | 其余球谐系数，$K=(d+1)^2$ |

**为什么不直接优化正尺度？** 无约束优化器可以把原始参数更新为任意实数；指数映射把渲染时的值限制为正。保存与读取参数时必须分清原始空间和激活后空间。这里的尺度对应标准差，协方差对应方差。


In [2]:
def sigmoid(x):
    # 本实验数值范围温和，直接使用定义式。
    return 1.0 / (1.0 + math.exp(-x))

def logit(p):
    assert 0 < p < 1, '不透明度必须严格位于零和一之间。'
    return math.log(p / (1 - p))

raw_scale = [math.log(0.1), math.log(0.2), math.log(0.4)]
scale = [math.exp(x) for x in raw_scale]
raw_opacity = logit(0.1)
assert math.isclose(sigmoid(raw_opacity), 0.1)
print('原始对数尺度：', [round(x, 4) for x in raw_scale])
print('渲染用正尺度：', [round(x, 4) for x in scale])
print('初始不透明度：', round(sigmoid(raw_opacity), 4))
print('三阶球谐每通道系数数目：', (3 + 1) ** 2)


原始对数尺度： [-2.3026, -1.6094, -0.9163]
渲染用正尺度： [0.1, 0.2, 0.4]
初始不透明度： 0.1
三阶球谐每通道系数数目： 16


## 三、尺度与旋转怎样产生协方差

源码：[数学辅助函数](../../utils/general_utils.py) 中的 `build_rotation`、`build_scaling_rotation`，以及模型中的 `get_covariance`。

采用列向量数学记号：$S=\operatorname{diag}(s)$，$L=RS$，$\Sigma=LL^T=RS^2R^T$。旋转改变椭球朝向，三个正尺度决定主轴长度。源码使用四元数，是为了用四个数表达旋转，并通过归一化约束到合法旋转。

下面实现与 Python 路径相同的数学关系。CUDA 中使用 GLM 的列存储约定，表面上出现 `S*R`、`transpose(M)*M`，不能不考虑存储和构造方式就判定公式写反。


In [3]:
def transpose(a):
    return [list(column) for column in zip(*a)]

def matmul(a, b):
    return [[sum(x * y for x, y in zip(row, column))
             for column in zip(*b)] for row in a]

def rotation_matrix(q):
    # 与源码相同，四元数按实部在前的顺序读取，并先归一化。
    norm = math.sqrt(sum(x * x for x in q))
    assert norm > 0, '零四元数不能表示旋转。'
    w, x, y, z = [v / norm for v in q]
    return [[1 - 2 * (y*y + z*z), 2 * (x*y - w*z), 2 * (x*z + w*y)],
            [2 * (x*y + w*z), 1 - 2 * (x*x + z*z), 2 * (y*z - w*x)],
            [2 * (x*z - w*y), 2 * (y*z + w*x), 1 - 2 * (x*x + y*y)]]

# 绕第三轴旋转四十五度，观察协方差出现非零交叉项。
angle = math.pi / 4
rotation = rotation_matrix([math.cos(angle / 2), 0, 0, math.sin(angle / 2)])
scale_matrix = [[scale[i] if i == j else 0 for j in range(3)] for i in range(3)]
linear = matmul(rotation, scale_matrix)
covariance = matmul(linear, transpose(linear))
assert all(math.isclose(covariance[i][j], covariance[j][i], abs_tol=1e-12)
           for i in range(3) for j in range(3))
assert math.isclose(sum(covariance[i][i] for i in range(3)), sum(s*s for s in scale))
print('世界坐标协方差：')
for row in covariance:
    print([round(x, 5) for x in row])
print('对称矩阵打包顺序：', [round(covariance[i][j], 5)
                             for i, j in [(0,0), (0,1), (0,2), (1,1), (1,2), (2,2)]])


世界坐标协方差：
[0.025, -0.015, 0.0]
[-0.015, 0.025, 0.0]
[0.0, 0.0, 0.16]
对称矩阵打包顺序： [0.025, -0.015, 0.0, 0.025, 0.0, 0.16]


## 四、初始化为何需要点云与近邻距离

源码：`GaussianModel.create_from_pcd`。点云位置成为初始均值，点云 RGB 编码为零阶球谐，高阶系数起初为零。四元数初始为单位旋转，不透明度初始为 0.1。

`distCUDA2` 来自 `simple-knn` 子模块；名称里的“2”不要理解成“两个最近邻”。当前实现计算三个近邻的**平方距离均值**。取平方根得到局部长度尺度，再取对数存入 `_scaling`。在稠密处用小高斯，在稀疏处用大高斯，比为整幅场景指定固定半径更合理。

下面用穷举实现相同的近邻统计概念，仅供几个点的推导；原扩展使用空间划分来加速。真实场景的数据规模不适合这段二次复杂度代码。


In [4]:
points = [(0, 0, 0), (0.1, 0, 0), (0, 0.2, 0), (0, 0, 0.3), (1, 1, 1)]
initial_scales = []
for i, p in enumerate(points):
    # 排除自身，排序后取三个近邻的平方距离均值。
    distances = sorted(sum((a-b)**2 for a, b in zip(p, q))
                       for j, q in enumerate(points) if i != j)
    mean_square = max(sum(distances[:3]) / 3, 1e-7)
    raw = math.log(math.sqrt(mean_square))
    initial_scales.append(math.exp(raw))
print('每个点的初始尺度：', [round(s, 4) for s in initial_scales])
assert initial_scales[-1] > initial_scales[0]
print('远离其他点的最后一个点获得更大的初始覆盖范围。')


每个点的初始尺度： [0.216, 0.2309, 0.2708, 0.3266, 1.6269]
远离其他点的最后一个点获得更大的初始覆盖范围。


## 五、球谐颜色为何随观察方向变化

源码：[球谐计算](../../utils/sh_utils.py) 的 `RGB2SH`、`SH2RGB`、`eval_sh`；渲染桥接中的 `dir_pp_normalized`。

零阶基函数是常数 $C_0=0.28209479177387814$。编码约定为 $f_0=(\mathrm{RGB}-0.5)/C_0$，渲染时加回 0.5。只使用零阶时，同一高斯从任何方向看都具有相同颜色。

一阶再加入方向分量，源码的基顺序与符号为 $-C_1 y, C_1 z, -C_1 x$，不是任意排列的 $x,y,z$。高阶可表达更复杂的视角变化，但它不是显式的材质/光照物理模型。

训练每 1000 步尝试增加活动阶数，最多达到设置的最大阶数。分配了十六个系数，不代表第一步就启用三阶。


In [5]:
c0, c1 = 0.28209479177387814, 0.4886025119029199
rgb = [0.2, 0.6, 0.8]
dc = [(v - 0.5) / c0 for v in rgb]
decoded = [c0 * v + 0.5 for v in dc]
assert all(math.isclose(a, b) for a, b in zip(rgb, decoded))
print('零阶颜色还原：', [round(v, 3) for v in decoded])

def first_order_red(direction):
    # 为红色通道设置一个非零的一阶系数，其他系数为零。
    x, y, z = direction
    return max(0, c0 * dc[0] - c1 * y * 0.3 + c1 * z * 0.0 - c1 * x * 0.0 + 0.5)

print('从正向第二轴观察的红色：', round(first_order_red((0, 1, 0)), 4))
print('从反向第二轴观察的红色：', round(first_order_red((0, -1, 0)), 4))


零阶颜色还原： [0.2, 0.6, 0.8]
从正向第二轴观察的红色： 0.0534
从反向第二轴观察的红色： 0.3466


## 六、相机坐标约定必须先讲清楚

源码：[数据读取](../../scene/dataset_readers.py)、[相机对象](../../scene/cameras.py)、[几何函数](../../utils/graphics_utils.py)。

数学上写世界到相机变换 $p_c=R_{cw}p_w+t_{cw}$；相机中心为 $C_w=-R_{cw}^Tt_{cw}$。这说明外参中的平移 $t_{cw}$ 一般**不是相机中心**。

COLMAP 数据使用自身的相机约定。合成数据的 `transform_matrix` 通常是相机到世界矩阵，读取器还需翻转对应轴，再求逆转为世界到相机。随后 Python 会转置一些矩阵以匹配光栅器存储约定。应区分：数学变换方向、向量左乘/右乘、内存存储布局，三者不是同一问题。

下面仅使用单位旋转验证平移关系，不复刻完整读取器。


In [6]:
camera_center = [1.0, 2.0, 3.0]
translation = [-v for v in camera_center]
world_point = [1.2, 2.1, 5.0]
camera_point = [p + t for p, t in zip(world_point, translation)]
assert camera_point[2] > 0
print('世界坐标点：', world_point)
print('相机坐标点：', [round(v, 4) for v in camera_point])
print('相机前方深度：', camera_point[2])
show_source('utils/graphics_utils.py', 'def getWorld2View2', before=2, after=14)


世界坐标点： [1.2, 2.1, 5.0]
相机坐标点： [0.2, 0.1, 2.0]
相机前方深度： 2.0
  52  # 在外参构造基础上，可对世界中的相机中心做平移与缩放，再返回新的世界到相机矩阵。
  53  # 默认 translate=0、scale=1 与普通外参一致；这不是自动把所有场景点一起归一化。
  54  def getWorld2View2(R, t, translate=np.array([.0, .0, .0]), scale=1.0):
  55      Rt = np.zeros((4, 4))
  56      Rt[:3, :3] = R.transpose()
  57      Rt[:3, 3] = t
  58      # 最后一行补成 [0,0,0,1]，将三维刚体变换写成 4×4 齐次矩阵。
  59      Rt[3, 3] = 1.0
  60  
  61      # 先求逆到相机到世界形式，才能直接读取/修改世界坐标中的相机中心。
  62      C2W = np.linalg.inv(Rt)
  63      cam_center = C2W[:3, 3]
  64      # 操作顺序是“先平移，再整体缩放”；旋转部分保持原值。
  65      # 若把这作为整个场景的坐标重参数化，场景几何也需由调用方采用相同约定。
  66      cam_center = (cam_center + translate) * scale
  67      C2W[:3, 3] = cam_center


## 七、三维协方差怎样投影成二维椭圆

源码：[前向内核](cuda-source/cuda_rasterizer/forward.cu) 的 `computeCov2D`、`preprocessCUDA`。

针孔投影为 $u=f_x x/z+c_x$、$v=f_y y/z+c_y$。在高斯中心附近线性化，得到

$$J=\begin{bmatrix}f_x/z&0&-f_xx/z^2\\0&f_y/z&-f_yy/z^2\end{bmatrix},\qquad
\Sigma_{2D}=J R_{cw}\Sigma R_{cw}^T J^T.$$

这是局部一阶近似，不是把三维高斯经过非线性透视变换后仍能得到一个严格高斯分布的证明。CUDA 还会限制用于雅可比计算的 $x/z,y/z$ 范围，缓和视场外极端值；下面选取靠近中心的点，因此不触发该限制。

当前内核始终向二维协方差对角线添加 0.3，保证最小像素覆盖。开启抗锯齿时再用行列式比例缩放不透明度，补偿低通扩张后的幅值。不能把“低通总是存在”和“抗锯齿幅值补偿开关”混为一谈。


In [7]:
width, height = 640, 480
fx = fy = 500.0
x, y, z = camera_point
jacobian = [[fx/z, 0, -fx*x/(z*z)], [0, fy/z, -fy*y/(z*z)]]
# 此例相机旋转为单位矩阵，因此可直接使用世界协方差。
cov2d = matmul(matmul(jacobian, covariance), transpose(jacobian))
det_before = cov2d[0][0] * cov2d[1][1] - cov2d[0][1] ** 2
filtered = [[cov2d[i][j] + (0.3 if i == j else 0) for j in range(2)] for i in range(2)]
a, b, c = filtered[0][0], filtered[0][1], filtered[1][1]
det_after = a*c - b*b
conic = [c/det_after, -b/det_after, a/det_after]
compensation = math.sqrt(max(0.000025, det_before / det_after))
mid = 0.5 * (a + c)
# 忠实保留此版本对特征值判别式的最小值保护。
largest = mid + math.sqrt(max(0.1, mid*mid - det_after))
radius = math.ceil(3 * math.sqrt(largest))
assert det_after > 0 and 0 < compensation <= 1
print('低通后的二维协方差：', [[round(v, 3) for v in row] for row in filtered])
print('二维逆协方差的三个独立元素：', [round(v, 7) for v in conic])
print('抗锯齿不透明度补偿系数：', round(compensation, 6))
print('近似三倍标准差包围圆半径：', radius, '像素')


低通后的二维协方差： [[1662.8, -887.5], [-887.5, 1587.8]]
二维逆协方差的三个独立元素： [0.0008571, 0.0004791, 0.0008976]
抗锯齿不透明度补偿系数： 0.999737
近似三倍标准差包围圆半径： 151 像素


## 八、为什么先分块，再排序

源码：[光栅调度](cuda-source/cuda_rasterizer/rasterizer_impl.cu) 的 `duplicateWithKeys`、`identifyTileRanges` 和 `CudaRasterizer::Rasterizer::forward`。

直接让每个像素遍历所有高斯，代价很高。实现把屏幕划成小块：先求每个高斯覆盖多少块，前缀和计算写入偏移，再为每个“高斯—块”重叠生成一条记录。排序键的高 32 位是块编号，低 32 位是正深度浮点数的位表示。这样一次基数排序同时按块聚类，并在块内按深度从近到远排序。

**同一个高斯可出现多次**，因为它可能覆盖多个块。因此 `num_rendered` 表示重叠记录数量，不能误当成原始高斯数或有效像素数。按中心深度排序是该实现的近似，不是对每个像素重新精确排序三维体积。


In [8]:
overlaps = [[0, 1], [1], [0, 1, 2]]
depths = [3.0, 1.0, 2.0]
prefix = []
total = 0
for touched in overlaps:
    total += len(touched)
    prefix.append(total)

def positive_float_bits(value):
    assert value > 0
    return struct.unpack('<I', struct.pack('<f', value))[0]

records = []
for gaussian_id, tiles in enumerate(overlaps):
    for tile in tiles:
        key = (tile << 32) | positive_float_bits(depths[gaussian_id])
        records.append((key, tile, gaussian_id, depths[gaussian_id]))
ordered = sorted(records)
assert len(ordered) == prefix[-1] == 6
print('覆盖数的包含式前缀和：', prefix)
print('排序后的（块编号，高斯编号，深度）：')
for _, tile, gaussian_id, depth in ordered:
    print(tile, gaussian_id, depth)
assert [g for _, tile, g, _ in ordered if tile == 1] == [1, 2, 0]


覆盖数的包含式前缀和： [2, 3, 6]
排序后的（块编号，高斯编号，深度）：
0 2 2.0
0 0 3.0
1 1 1.0
1 2 2.0
1 0 3.0
2 2 2.0


## 九、一个像素怎样融合多个高斯

源码：[前向内核](cuda-source/cuda_rasterizer/forward.cu) 的 `renderCUDA`。

令二维像素偏移为 $d$，逆协方差为 $Q$，则局部高斯值 $G=\exp(-\frac12d^TQd)$，有效透明度 $\alpha=\min(0.99,oG)$。从近到远处理：

$$T_1=1,\quad w_i=T_i\alpha_i,\quad C=\sum_iw_ic_i+T_{\mathrm{end}}c_{bg},\quad T_{i+1}=T_i(1-\alpha_i).$$

当前代码略过 $\alpha<1/255$ 的小贡献；如果加入当前高斯后透射率会低于 $10^{-4}$，直接结束该像素，**触发终止的当前高斯也没有被加入累积**。这是阅读实际代码才容易注意到的顺序细节。

颜色权重不需要额外归一化：剩余权重交给背景。换高斯顺序一般会改变颜色，因此排序有实际意义。


In [9]:
def composite(alphas, colors, depths, background=(1.0, 1.0, 1.0)):
    # 模拟单个像素；输入透明度已包含高斯衰减，未包含前方透射率。
    transmittance = 1.0
    result = [0.0, 0.0, 0.0]
    inverse_depth = 0.0
    weights = []
    for alpha, color, depth in zip(alphas, colors, depths):
        alpha = min(0.99, alpha)
        if alpha < 1/255:
            continue
        remaining = transmittance * (1 - alpha)
        if remaining < 0.0001:
            break
        weight = transmittance * alpha
        weights.append(weight)
        result = [v + weight*c for v, c in zip(result, color)]
        inverse_depth += weight / depth
        transmittance = remaining
    result = [v + transmittance*b for v, b in zip(result, background)]
    return result, weights, transmittance, inverse_depth

colors = [(1, 0, 0), (0, 0, 1)]
pixel, weights, remaining, inverse_depth = composite([0.5, 0.8], colors, [2, 4])
assert all(math.isclose(a, b) for a, b in zip(pixel, [0.6, 0.1, 0.5]))
assert math.isclose(sum(weights) + remaining, 1)
print('两个高斯的贡献权重：', weights)
print('最后剩余透射率：', round(remaining, 4))
print('像素颜色：', [round(v, 4) for v in pixel])
reversed_pixel = composite([0.8, 0.5], list(reversed(colors)), [4, 2])[0]
print('交换顺序后的颜色：', [round(v, 4) for v in reversed_pixel])


两个高斯的贡献权重： [0.5, 0.4]
最后剩余透射率： 0.1
像素颜色： [0.6, 0.1, 0.5]
交换顺序后的颜色： [0.2, 0.1, 0.9]


## 十、用字符图观察二维高斯的覆盖

下面只画二维核，不执行分块、相机投影或颜色混合。横向尺度较大时，核变宽；交叉项可以使椭圆倾斜。这是查看“点为什么变成一片”的最小实验。

读源码时要把三种量分开：三维世界尺度、投影后的二维协方差、用于快速找覆盖块的像素包围半径。它们的单位和用途不同。


In [10]:
levels = ' ·░▒▓█'
for y in range(-6, 7):
    line = ''
    for x in range(-14, 15):
        # 该正定二次型包含交叉项，因此图案呈倾斜椭圆。
        power = -0.5 * (x*x/25 + y*y/4 - 0.1*x*y)
        value = math.exp(power)
        line += levels[min(len(levels)-1, int(value * len(levels)))]
    print(line)


                             
                             
      ·······                
    ···░░░░░░░····           
    ··░░▒▒▒▓▓▒▒▒░░··         
    ··░░▒▓▓████▓▓▒▒░··       
     ··░░▒▓███████▓▒░░··     
       ··░▒▒▓▓████▓▓▒░░··    
         ··░░▒▒▒▓▓▒▒▒░░··    
           ····░░░░░░░···    
                ·······      
                             
                             


## 十一、名为深度的输出为什么是逆深度

这里的 $z_i$ 是相机坐标系的第三分量，不是相机到高斯中心的欧氏距离。当前 CUDA 累积的是 $D_{inv}=\sum_i w_i/z_i$，背景没有额外逆深度贡献，也没有除以 $\sum_iw_i$。所以它既不是 $\sum_iw_iz_i$，也不是归一化后的平均逆深度。

这影响低不透明度区域的解释：即便高斯中心的相机坐标 z 分量不变，透明度降低也会降低该输出。不能直接将 `1 / depth_image` 当作可靠的米制深度图。

训练器的单目监督也使用经过尺度/偏移处理的逆深度。掩码乘在误差上，然后对整张图取平均；无效区域仍占分母。


In [11]:
normalized_inverse = inverse_depth / sum(weights)
weighted_depth = sum(w*z for w, z in zip(weights, [2, 4]))
print('源码输出的未归一化逆深度：', round(inverse_depth, 6))
print('按总不透明度归一化后的逆深度：', round(normalized_inverse, 6))
print('直接累积普通深度：', round(weighted_depth, 6))
assert math.isclose(inverse_depth, 0.35)

errors, mask = [1.0, 1.0, 9.0, 9.0], [1, 1, 0, 0]
full_mean = sum(abs(e*m) for e, m in zip(errors, mask)) / len(errors)
valid_mean = sum(abs(e*m) for e, m in zip(errors, mask)) / sum(mask)
print('源码式全图平均误差：', full_mean)
print('只在有效区域平均的误差：', valid_mean)
assert full_mean != valid_mean


源码输出的未归一化逆深度： 0.35
按总不透明度归一化后的逆深度： 0.388889
直接累积普通深度： 2.6
源码式全图平均误差： 0.5
只在有效区域平均的误差： 1.0


## 十二、反向传播先从一个像素推导

源码：[反向内核](cuda-source/cuda_rasterizer/backward.cu) 的 `renderCUDA`，以及 [自动微分桥接](cuda-source/diff_gaussian_rasterization/__init__.py) 的 `_RasterizeGaussians.backward`。

对两个标量颜色，$C=a_1c_1+(1-a_1)a_2c_2+(1-a_1)(1-a_2)b$。因此

$$\frac{\partial C}{\partial a_1}=c_1-a_2c_2-(1-a_2)b,\qquad
\frac{\partial C}{\partial a_2}=(1-a_1)(c_2-b).$$

一个高斯的透明度改变，不只改变自身贡献，也改变后方高斯和背景的可见程度。反向内核逆序遍历前向用过的贡献，利用保存的最终透射率和最后贡献位置重建中间量，避免为每像素每高斯存全部状态。

下面用中心差分验证光滑区间的解析导数。它只验证这条公式，没有验证 CUDA 实现、阈值分支、排序变化或增密这些不连续操作。尤其当前 CUDA 对透明度上限使用的反传不是严格逐分支的 `min` 导数。


In [12]:
def two_layer_color(a1, a2, c1=0.8, c2=0.3, background=0.1):
    return a1*c1 + (1-a1)*a2*c2 + (1-a1)*(1-a2)*background

a1, a2, epsilon = 0.4, 0.6, 1e-6
analytic1 = 0.8 - a2*0.3 - (1-a2)*0.1
analytic2 = (1-a1)*(0.3-0.1)
numeric1 = (two_layer_color(a1+epsilon, a2)-two_layer_color(a1-epsilon, a2))/(2*epsilon)
numeric2 = (two_layer_color(a1, a2+epsilon)-two_layer_color(a1, a2-epsilon))/(2*epsilon)
assert math.isclose(analytic1, numeric1, rel_tol=1e-7)
assert math.isclose(analytic2, numeric2, rel_tol=1e-7)
print('前方透明度导数：', round(analytic1, 6), '；中心差分：', round(numeric1, 6))
print('后方透明度导数：', round(analytic2, 6), '；中心差分：', round(numeric2, 6))


前方透明度导数： 0.58 ；中心差分： 0.58
后方透明度导数： 0.12 ；中心差分： 0.12


## 十三、二维均值梯度怎样返回训练器

渲染桥接创建全零 `screenspace_points` 并调用 `retain_grad()`。这个张量是接收梯度的接口占位，不是已经算好的真实投影坐标，也不是另一个独立优化的几何参数。真实中心由 CUDA 根据三维位置和相机计算。

自动微分桥接在 `backward` 中把 CUDA 算出的二维均值梯度放回 `means2D` 对应的返回位置。训练器读取它前两维梯度的范数，累计到每个高斯上。其余梯度则继续流向三维位置、协方差、球谐系数、尺度与旋转。

返回的前两维梯度在 CUDA 中还乘了 0.5×图像宽度、0.5×图像高度，对应归一化屏幕坐标的尺度；不要把它直接解释成“每移动一个像素”的导数，也不要据此随意迁移增密阈值。

多个像素会对同一个高斯贡献梯度，CUDA 使用 `atomicAdd` 累加。这也解释了为什么即便固定随机种子，浮点原子累积顺序仍可能造成细微非确定性。

**断点建议**：依次查看 `screenspace_points.shape`、`radii.shape`、`loss`、`screenspace_points.grad.shape` 和 `xyz_gradient_accum`。必须在 `loss.backward()` 之后检查梯度。


In [13]:
# 模拟同一个高斯在三次可见观测中的屏幕平面梯度。
screen_gradients = [(0.0003, 0.0004), (0.0006, 0.0008), (0.0, 0.0)]
accumulated = sum(math.hypot(gx, gy) for gx, gy in screen_gradients)
observations = len(screen_gradients)
average_gradient = accumulated / observations
print('梯度范数累计值：', round(accumulated, 7))
print('可见观测次数：', observations)
print('用于增密筛选的平均值：', round(average_gradient, 7))
assert average_gradient > 0.0002
show_source('scene/gaussian_model.py', 'def add_densification_stats', before=3, after=10)


梯度范数累计值： 0.0015
可见观测次数： 3
用于增密筛选的平均值： 0.0005
 725  
 726      # 在 loss.backward() 后收集稠密化信号；渲染器的屏幕点张量必须保留梯度。
 727      # 当前渲染器传入 [K,1] 整数索引；本方法也能接收 [N] 布尔掩码。被选点仍可能因遮挡得到零梯度。
 728      def add_densification_stats(self, viewspace_point_tensor, update_filter):
 729          # 取屏幕均值梯度的前两维 (x,y) 并求范数。当前 [K,1] 索引得到 [K,1,2]，范数为 [K,1,1]，
 730          # 与左侧高级索引结果匹配；若改用一维布尔掩码，则对应形状为 [K,2] 与 [K,1]。
 731          # 先逐视角取范数再累加，可避免不同视角梯度方向互相抵消；不是累加三维位置梯度。
 732          self.xyz_gradient_accum[update_filter] += torch.norm(viewspace_point_tensor.grad[update_filter,:2], dim=-1, keepdim=True)
 733          # 每个本轮有效点计数加 1，之后与梯度累积相除得到跨视角平均强度。
 734          self.denom[update_filter] += 1


## 十四、克隆、分裂、裁剪分别解决什么

源码：`densify_and_clone`、`densify_and_split`、`densify_and_prune`。

高梯度表示当前位置附近可能缺少表达能力，再根据最大三维尺度分两种情况：小高斯克隆，父点保留；大高斯分裂，按椭球局部尺度随机采样子点、旋转到世界坐标，再删除父点。分裂子点尺度除以 $0.8N$，这是实现采用的启发式，不应当解释成严格质量守恒定理。

低不透明度高斯被裁剪。启用尺寸检查时还考虑屏幕半径和世界尺度。增密会增加参数对象的行数，Adam 的一阶/二阶矩状态也必须同步扩展；裁剪时则用相同掩码切片。新增行的状态从零开始，不能把旧张量不加处理地塞回优化器。

**原版细节**：增密辅助函数会重置统计数组，某些后续尺寸判定读取到的是重置后的值；应按实际调用顺序理解。本文不修复或改写这些行为。


In [14]:
scene_extent, percent_dense, grad_threshold = 2.0, 0.01, 0.0002
examples = [
    {'编号': 0, '梯度': 0.0005, '尺度': 0.005, '不透明度': 0.5},
    {'编号': 1, '梯度': 0.0005, '尺度': 0.08, '不透明度': 0.5},
    {'编号': 2, '梯度': 0.0001, '尺度': 0.01, '不透明度': 0.002},
    {'编号': 3, '梯度': 0.0001, '尺度': 0.01, '不透明度': 0.5},
]
for item in examples:
    # 这里只展示各条件的概念分类，不复刻原实现先增密后裁剪的状态变更。
    if item['不透明度'] < 0.005:
        action = '裁剪'
    elif item['梯度'] >= grad_threshold:
        action = '克隆并保留父点' if item['尺度'] <= percent_dense*scene_extent else '分裂并删除父点'
    else:
        action = '保留并继续优化'
    print('高斯', item['编号'], '：', action)

# 复刻分裂中的局部采样和尺度缩小；固定种子便于观察。
rng = random.Random(7)
child_local_offsets = [[rng.gauss(0, s) for s in [0.08, 0.04, 0.02]] for _ in range(2)]
print('两个子点的局部偏移：', [[round(v, 4) for v in row] for row in child_local_offsets])
print('分裂后最大尺度：', 0.08 / (0.8 * 2))

# 参数数目变化时，一阶矩与二阶矩必须采用相同的扩展/裁剪方式。
first_moment = [0.1, 0.2, 0.3]
extended_moment = first_moment + [0.0, 0.0]
keep = [True, False, True, True, True]
pruned_moment = [value for value, selected in zip(extended_moment, keep) if selected]
assert pruned_moment == [0.1, 0.3, 0.0, 0.0]
print('扩展并裁剪后的一阶矩：', pruned_moment)


高斯 0 ： 克隆并保留父点
高斯 1 ： 分裂并删除父点
高斯 2 ： 裁剪
高斯 3 ： 保留并继续优化
两个子点的局部偏移： [[-0.0205, 0.0205, -0.0045], [-0.0252, -0.0372, -0.0043]]
分裂后最大尺度： 0.049999999999999996
扩展并裁剪后的一阶矩： [0.1, 0.3, 0.0, 0.0]


## 十五、图像损失与学习率

源码：[损失函数](../../utils/loss_utils.py)、[训练循环](../../train.py)、[调度函数](../../utils/general_utils.py)。

总图像损失为 $(1-\lambda)L_1+\lambda(1-\mathrm{SSIM})$。本代码直接用 $1-\mathrm{SSIM}$，没有另加二分之一。局部 SSIM 通过窗口估计均值、方差与协方差，它关心局部结构；不能把整幅图像的单个全局均值替代滑动窗口，声称复现了源码 SSIM。

位置学习率在对数空间插值：$\eta(t)=\exp((1-t)\log\eta_0+t\log\eta_1)$。因此训练中点是几何平均，而非算术平均。位置学习率还乘场景尺度；曝光另有独立调度。

下面仅验证加权组合与指数调度，SSIM 值是假设的已计算值。


In [15]:
prediction, target = [0.2, 0.5, 0.9], [0.1, 0.6, 0.7]
l1 = sum(abs(p-t) for p, t in zip(prediction, target)) / len(target)
lambda_dssim, assumed_ssim = 0.2, 0.8
loss = (1-lambda_dssim)*l1 + lambda_dssim*(1-assumed_ssim)
print('平均绝对误差：', round(l1, 6))
print('加权图像损失：', round(loss, 6))

def exponential_schedule(step, initial=0.00016, final=0.0000016, total=30000):
    # 此小实验省略延迟预热和场景尺度，只展示指数插值本身。
    t = min(1, max(0, step/total))
    return math.exp((1-t)*math.log(initial) + t*math.log(final))

for step in [0, 7500, 15000, 30000]:
    print('迭代', step, '：位置学习率', f'{exponential_schedule(step):.8f}')
assert math.isclose(exponential_schedule(15000), math.sqrt(0.00016*0.0000016))


平均绝对误差： 0.133333
加权图像损失： 0.146667
迭代 0 ：位置学习率 0.00016000
迭代 7500 ：位置学习率 0.00005060
迭代 15000 ：位置学习率 0.00001600
迭代 30000 ：位置学习率 0.00000160


## 十六、曝光补偿与训练区域

模型为训练相机维护 $3\times4$ 曝光参数，由颜色变换矩阵和偏移组成。渲染器先得到图像，再执行颜色仿射变换。初始单位矩阵与零偏移不改变图像。它可吸收拍摄曝光变化，但不是给每个高斯再添加一套颜色。

当前训练/测试曝光模式会使用图像半幅策略；相机类中的掩码与评估时右半幅切片应一起阅读。`capture()` 的检查点元组没有包含曝光参数与其优化器，不能据此保证曝光训练的精确续训。PLY 场景导出与训练检查点也不是同一种文件。


In [16]:
# 按渲染器的行向量右乘约定演示一个像素的颜色变换。
input_color = [0.2, 0.4, 0.6]
exposure_matrix = [[1.1, 0, 0], [0, 0.9, 0], [0, 0, 1.0]]
exposure_bias = [0.01, 0.02, -0.01]
transformed = matmul([input_color], exposure_matrix)[0]
transformed = [min(1, max(0, v+b)) for v, b in zip(transformed, exposure_bias)]
print('曝光补偿后的颜色：', [round(v, 4) for v in transformed])
assert all(math.isclose(a, b) for a, b in zip(transformed, [0.23, 0.38, 0.59]))


曝光补偿后的颜色： [0.23, 0.38, 0.59]


## 十七、一次训练迭代的真实顺序

回到 [train.py](../../train.py)，按以下顺序对照代码：

1. 可选处理查看器请求，启动本轮 GPU 计时。
2. 更新学习率，必要时提升活动球谐阶数。
3. 从相机栈不放回随机取一个相机，渲染 RGB 与逆深度。
4. 应用图像掩码，计算图像损失与可选逆深度损失。
5. `loss.backward()`，让参与当前损失的参数与屏幕坐标占位张量获得梯度。
6. 在 `no_grad` 中记录日志、评估、按配置导出场景。
7. 在增密阶段累计统计，按间隔克隆/分裂/裁剪，必要时重置不透明度。
8. 未到最终迭代时，先更新曝光优化器，再更新高斯优化器，清空梯度。
9. 按配置保存检查点。

**三个容易忽略的顺序问题**：场景导出发生在本轮增密和优化之前；检查点保存发生在它们之后；最后一轮不执行优化器更新。发生增密/重置时，参数对象可能已经替换，不能简单假设本轮所有梯度仍附着在相同的对象上。

下面直接从源码抽取默认值，避免在笔记里长期维护第二套容易过期的参数表。


In [17]:
# 用语法树读取常量，不导入训练模块，也不执行类构造函数。
parameter_tree = ast.parse((repo / 'arguments/__init__.py').read_text(encoding='utf-8'))
optimization_class = next(n for n in parameter_tree.body
                          if isinstance(n, ast.ClassDef) and n.name == 'OptimizationParams')
defaults = {}
for node in ast.walk(optimization_class):
    if isinstance(node, ast.Assign) and len(node.targets) == 1:
        target_node = node.targets[0]
        if isinstance(target_node, ast.Attribute) and isinstance(node.value, ast.Constant):
            defaults[target_node.attr] = node.value.value
labels = {'iterations': '总迭代数', 'densify_from_iter': '增密预热结束',
          'densify_until_iter': '增密结束', 'densification_interval': '增密间隔',
          'opacity_reset_interval': '不透明度重置间隔', 'lambda_dssim': '结构损失权重'}
for key, label in labels.items():
    print(label, '：', defaults[key])
assert defaults['densify_from_iter'] < defaults['densify_until_iter'] < defaults['iterations']


总迭代数 ： 30000
增密预热结束 ： 500
增密结束 ： 15000
增密间隔 ： 100
不透明度重置间隔 ： 3000
结构损失权重 ： 0.2


## 十八、从 Python 追到 CUDA 的调用链

按顺序打开下面的文件，并在每个函数处回答“输入、输出、保存的中间量分别是什么”：

| 层次 | 入口 | 阅读目标 |
|---|---|---|
| 训练 | `training` | 渲染结果怎样变成损失 |
| 渲染桥接 | `render` | 模型参数怎样激活并打包 |
| Python 扩展包装 | `GaussianRasterizer.forward` | 互斥输入怎样检查，空参数怎样表示 |
| 自动微分 | `_RasterizeGaussians.forward/backward` | 中间缓存如何保存，梯度按什么顺序返回 |
| 扩展入口 | `ext.cpp`、`rasterize_points.cu` | 张量如何转为指针，输出如何分配 |
| 调度 | `rasterizer_impl.cu` | 预处理、前缀和、复制、排序、像素计算 |
| 前向 | `forward.cu` | 球谐、投影、椭圆、透明度融合 |
| 反向 | `backward.cu` | 逆序融合、协方差/位置/旋转链式求导 |

主仓库中的 `cuda-source` 是逐版本对应的中文学习副本，**不会被安装脚本自动编译**。实际构建仍使用原子模块。需要在实际子模块中阅读相同注释时，可按中文指南应用随附补丁；执行代码保持一致。


## 十九、在有 CUDA 的环境中继续实验

这份笔记的所有代码单元都能在 CPU 运行。原项目训练需要另行准备匹配的 PyTorch、CUDA 编译器、光栅器和近邻扩展，以及经过相机标定的数据。不要因为本笔记通过就认为原生 GPU 训练已经通过。

准备环境后，可在分配到 GPU 的计算节点上按如下顺序做小规模冒烟测试。命令中的路径是需要替换的占位值，本文不会自动提交集群作业：

```bash
# 先确认运行环境能看到分配到的显卡。
python -c "import torch; print(torch.cuda.is_available())"

# 用少量迭代确认数据读取、渲染、反传和场景导出连通。
# 一百轮低于默认增密起点，因此这一步不验证增密分支。
python train.py -s /数据集路径 -m /实验输出路径 --iterations 100 --resolution 8 --disable_viewer --test_iterations 100 --save_iterations 100

# 导出已保存检查点的图像；数据划分以训练配置为准。
python render.py -m /实验输出路径 --iteration 100
```

如果要专门观察增密，应再进行一个超过预热期的独立小实验，同时监控高斯数量与显存。不必立即做论文规模训练。hsg 登录/编辑节点本身不能代替 GPU 计算节点；GPU 测试需要遵循集群调度规则。

本轮实际完成的检查及尚未运行的 GPU 项目记录在 [验证记录](VALIDATION_zh.md)。


## 二十、自测与下一轮精读

请先口头解释，再回到代码找出准确的行：

1. 为什么 `_scaling` 可以是负数，但 `get_scaling` 不会是负数？
2. 为什么四元数的第一项在这里是实部，而不能套用任意库的默认顺序？
3. `world_view_transform` 与 `full_proj_transform` 分别包含哪些变换？
4. 把 `screenspace_points` 初始化为零，为什么仍能读取有意义的梯度？
5. 一个高斯覆盖多个块时，排序缓冲区会有几条记录？
6. 透明度为什么要乘当前透射率？背景权重从哪里来？
7. `depth` 为什么不能直接解释为米制深度？
8. 克隆为什么保留父点，分裂为什么删除父点？
9. 改变高斯数量时，Adam 的状态为什么也必须改变？
10. 最终迭代的 PLY 与同轮检查点为什么可能不是同一时刻的状态？

建议每次只选一个函数：在纸上写出输入形状，逐个追踪中间张量，再根据返回值找到调用者。中文注释负责补充意图与约定，真正的边界条件仍以紧邻的表达式为准。
